### Handling JSON-data in pandas (both simple and complex structures)

In [91]:
import json
import numpy as np 
import pandas as pd 
import seaborn as sns

In [92]:
# if the data is in optimal format (JSON -> list of dictionries, but no nested structures)
# this can be done easily (COMPARE this to pd.read_csv() )
df = pd.read_json("testing.json")

# show the first 5 only
df.head(5)

,id,name,value
0,1,Pannier,297
1,2,Voltsillam,388
2,3,Rank,501
3,4,Toughjoyfax,281
4,5,Cardify,319


**Traditionally in Python, this is something like this:**

In [93]:
# open JSON file and process it
json_file = open("testing.json", "r")
json_data = json_file.read()
data = json.loads(json_data)
json_file.close()

# loop through the data => list of dictionaries
for row in data:
    print(row['name'])

Pannier
Voltsillam
Rank
Toughjoyfax
Cardify
Job
Fintone
Cardguard
Stronghold
Transcof
Voltsillam
Holdlamis
Voyatouch
Matsoft
Duobam
Quo Lux
Holdlamis
Toughjoyfax
Mat Lam Tam
Tempsoft
Fix San
Aerified
Flexidy
Ronstring
Matsoft
Subin
Stringtough
Rank
Sonsing
Alpha
Y-Solowarm
Matsoft
It
Gembucket
Sonsing
Domainer
Ventosanzap
Trippledex
Tempsoft
Viva
Tres-Zap
Cardify
Fix San
Lotstring
Voltsillam
Voltsillam
Aerified
Tampflex
Alphazap
Y-find
Pannier
Gembucket
Stim
Bitchip
Latlux
Prodder
Bytecard
Zoolab
Aerified
Tin
Zoolab
Prodder
Job
Zamit
Zontrax
Stringtough
Lotstring
Gembucket
Bytecard
Veribet
Ventosanzap
Sonsing
Andalax
Stronghold
Namfix
Duobam
Biodex
Prodder
Aerified
Sonair
Fix San
Treeflex
Ventosanzap
Treeflex
Otcom
Greenlam
Fixflex
Tampflex
Holdlamis
Rank
Regrant
Lotlux
Prodder
Latlux
Sonsing
Opela
Viva
Kanlam
Daltfresh
Cookley


### Let's try another JSON-file

In [94]:
df2 = pd.read_json("somedata.json")
df2

,id,name,contact,activities
0,1,Rovaniemi,"{'phone': '+358123456789', 'email': 'visit@rov...","[nature, arctic, safaris]"
1,2,Helsinki,"{'phone': '+358987654321', 'email': 'info@hels...","[culture, nightlife, restaurant]"


### Let's try pandas normalization -features

In [95]:
# open the JSON-file and process it with json-module
json_file = open("somedata.json")
response_raw = json_file.read()
json_file.close()
response = json.loads(response_raw)

# use pandas normalize() and create a DataFrame
cities = pd.json_normalize(response)

In [96]:
cities

,id,name,activities,contact.phone,contact.email
0,1,Rovaniemi,"[nature, arctic, safaris]",+358123456789,visit@rovaniemi.com
1,2,Helsinki,"[culture, nightlife, restaurant]",+358987654321,info@helsinki.fi


### How to process the activities-list?

In [97]:
# option 1: leave the list as it is, but remove the brackets etc.

# old-school way of replacing all the unneeded stuff
# cities['activities'] = cities['activities'].astype(str)
# cities['activities'] = cities['activities'].str.replace("[", "")
# cities['activities'] = cities['activities'].str.replace("]", "")
# cities['activities'] = cities['activities'].str.replace('"', "")
# cities['activities'] = cities['activities'].str.replace("'", "")

# given by ChatGPT, we can also apply lambda for the same effect
cities['activities'] = cities['activities'].apply(lambda x: ', '.join(x))

In [98]:
cities

,id,name,activities,contact.phone,contact.email
0,1,Rovaniemi,"nature, arctic, safaris",+358123456789,visit@rovaniemi.com
1,2,Helsinki,"culture, nightlife, restaurant",+358987654321,info@helsinki.fi


In [99]:
# option 2 => use the json_normalize with more details/parameters
# record_path => all the lists inside each dictionary
# meta = all basic variables and other dictionaries / objects
# the record_path and meta HAVE TO MATCH the original JSON data structure
cities_new = pd.json_normalize(
    response,
    record_path=['activities'],
    meta=['name',
          ['contact', 'phone'],
          ['contact', 'email']],
          errors="ignore"
)

In [100]:
cities_new

,0,name,contact.phone,contact.email
0,nature,Rovaniemi,+358123456789,visit@rovaniemi.com
1,arctic,Rovaniemi,+358123456789,visit@rovaniemi.com
2,safaris,Rovaniemi,+358123456789,visit@rovaniemi.com
3,culture,Helsinki,+358987654321,info@helsinki.fi
4,nightlife,Helsinki,+358987654321,info@helsinki.fi
5,restaurant,Helsinki,+358987654321,info@helsinki.fi


In [101]:
# pip install flatten_json

# OPTION 3 -> use flatten_json to separate each activity into separate columns
# let's use the original response again
from flatten_json import flatten

# flatten the original data and create a DataFrame
dict_flattened = (flatten(record, '.') for record in response)
df_flattened = pd.DataFrame(dict_flattened)

In [102]:
df_flattened

,id,name,contact.phone,contact.email,activities.0,activities.1,activities.2
0,1,Rovaniemi,+358123456789,visit@rovaniemi.com,nature,arctic,safaris
1,2,Helsinki,+358987654321,info@helsinki.fi,culture,nightlife,restaurant
